# 03 — Robustez a sesión, sesgo temporal y sensibilidad de los baselines

## Proyecto
**Kaki_DeepHS_StorageDays**

## Pregunta central

El notebook 02 mostró que, con validación estrictamente `fruit-disjoint`, el mejor baseline fue **Ridge sobre VIS_COR**, con buen desempeño OOF.

Sin embargo, en esta campaña experimental:

\[
\text{storage day} \leftrightarrow \text{acquisition session}
\]

Por tanto, separar frutos entre train/test evita fuga de identidad, pero no separa completamente el efecto fisiológico del almacenamiento del posible efecto instrumental/de sesión.

Este notebook no intenta "resolver" causalmente esa confusión —eso requeriría otra campaña o sesiones cruzadas—, sino someter el resultado a **pruebas de estrés**.

## Análisis principales

1. **Session-excluded fruit-disjoint stress test**  
   Para predecir el día \(d\), se eliminan del entrenamiento todas las observaciones adquiridas en el mismo `storage_day/session = d`.

2. **Baseline de intensidad global**  
   Un modelo lineal usa únicamente la media espectral de cada fruto-día. Si una sola medida global explica casi todo, puede existir una fuerte componente de amplitud global.

3. **Sensibilidad al desbalance temporal**  
   Se reentrena Ridge con:
   - ponderación por día;
   - ponderación conjunta fruto×día.

4. **Sesgo por día**  
   Se analiza especialmente la subestimación de los días 7–8.

5. **Comparaciones pareadas por fruto**
   - VIS vs VIS_COR;
   - Ridge vs PLSR;
   - Ridge estándar vs session-excluded;
   - Ridge estándar vs versiones ponderadas.

## Advertencia metodológica importante

La validación `session-excluded` es una prueba de estrés más exigente, pero **no es un experimento causal puro**. Al excluir el día objetivo también se excluye esa etiqueta del entrenamiento:

- los días internos se predicen por interpolación;
- día 0 y día 8 requieren extrapolación.

Por tanto, una degradación de desempeño puede reflejar simultáneamente:
- dependencia de sesión;
- dificultad de interpolar/extrapolar a un día no observado.

> No se entrenan CNN en este notebook.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import math
import os
import platform
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "03_kaki_robustness_session_bias"
RUN_SIGNATURE = "robustness_session_v1_maskV2"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

BASELINE_RESULTS = ROOT / "results/02_kaki_fruit_disjoint_splits_baselines"

RESULTS = ROOT / "results/03_kaki_robustness_session_bias"
FIGURES = ROOT / "figures/03_kaki_robustness_session_bias"
TABLES = ROOT / "tables/03_kaki_robustness_session_bias"
EXPORT_ROOT = ROOT / "exports/03_kaki_robustness_session_bias"

CHECKPOINT_DIR = RESULTS / f"checkpoints_{RUN_SIGNATURE}"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
    CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260818
OUTER_FOLDS = 5
INNER_FOLDS = 4
BOOTSTRAP_REPS = 5000

# Reutilizar checkpoints si Colab se desconecta.
RESUME = True

print("ROOT:", ROOT)
print("RUN_SIGNATURE:", RUN_SIGNATURE)


## 2. Cargar espectros aprobados MASK V2


In [ ]:
cube_cache_paths = {
    "VIS": INTERIM / "kaki_cube_spectra_VIS.csv",
    "VIS_COR": INTERIM / "kaki_cube_spectra_VIS_COR.csv",
}

spectra_paths = {
    "VIS": INTERIM / "kaki_fruit_day_spectra_VIS.csv",
    "VIS_COR": INTERIM / "kaki_fruit_day_spectra_VIS_COR.csv",
}

expected_cube_counts = {
    "VIS": 395,
    "VIS_COR": 398,
}

mask_gate_rows = []
spectral_data = {}
wavelengths = {}

for camera in ["VIS", "VIS_COR"]:

    cube_path = cube_cache_paths[camera]
    spec_path = spectra_paths[camera]

    if not cube_path.exists():
        raise FileNotFoundError(cube_path)

    if not spec_path.exists():
        raise FileNotFoundError(spec_path)

    cube_df = pd.read_csv(cube_path)

    median_coverage = float(
        cube_df["mask_coverage"].median()
    )

    mask_gate_rows.append({
        "camera_type": camera,
        "n_cubes": int(len(cube_df)),
        "median_mask_coverage": median_coverage,
        "n_mask_qc_not_ok": int(
            (cube_df["mask_qc_flag"] != "ok").sum()
        ),
        "n_extraction_errors": int(
            cube_df["extraction_error"].notna().sum()
        ),
    })

    assert len(cube_df) == expected_cube_counts[camera]
    assert median_coverage < 0.87
    assert (cube_df["mask_qc_flag"] == "ok").all()
    assert cube_df["extraction_error"].isna().all()

    df = pd.read_csv(spec_path)

    df["fruit_id"] = pd.to_numeric(
        df["fruit_id"],
        errors="raise"
    ).astype(int)

    df["storage_days_model"] = pd.to_numeric(
        df["storage_days_model"],
        errors="raise"
    ).astype(int)

    wl_cols = [
        c for c in df.columns
        if c.startswith("wl_")
    ]

    assert len(df) == 199
    assert df["fruit_id"].nunique() == 40
    assert not df[wl_cols].isna().any().any()

    spectral_data[camera] = (
        df.sort_values(
            ["fruit_id", "storage_days_model"]
        )
        .reset_index(drop=True)
    )

    wavelengths[camera] = np.array(
        [
            float(c.replace("wl_", ""))
            for c in wl_cols
        ],
        dtype=float
    )

mask_gate = pd.DataFrame(mask_gate_rows)

display(mask_gate)

mask_gate.to_csv(
    TABLES / "01_mask_v2_gate.csv",
    index=False
)

print("✅ MASK V2 confirmada.")


## 3. Cargar splits congelados y resultados OOF del notebook 02


In [ ]:
OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

OOF_PATH = (
    BASELINE_RESULTS /
    "kaki_oof_predictions.csv"
)

SELECTED_CONFIGS_PATH = (
    BASELINE_RESULTS /
    "kaki_selected_configs.csv"
)

for p in [
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
    OOF_PATH,
    SELECTED_CONFIGS_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo del notebook 02: {p}"
        )

outer_splits = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner_splits = pd.read_csv(
    INNER_SPLIT_PATH
)

baseline_oof = pd.read_csv(
    OOF_PATH
)

selected_configs = pd.read_csv(
    SELECTED_CONFIGS_PATH
)

for df in [
    outer_splits,
    inner_splits,
    baseline_oof,
    selected_configs,
]:
    if "fruit_id" in df.columns:
        df["fruit_id"] = pd.to_numeric(
            df["fruit_id"],
            errors="raise"
        ).astype(int)

outer_splits["outer_fold"] = pd.to_numeric(
    outer_splits["outer_fold"],
    errors="raise"
).astype(int)

inner_splits["outer_fold"] = pd.to_numeric(
    inner_splits["outer_fold"],
    errors="raise"
).astype(int)

inner_splits["inner_fold"] = pd.to_numeric(
    inner_splits["inner_fold"],
    errors="raise"
).astype(int)

assert outer_splits["fruit_id"].nunique() == 40
assert outer_splits["outer_fold"].nunique() == 5

print("✅ Splits y OOF del notebook 02 cargados.")


## 4. Auditoría anti-leakage heredada


In [ ]:
all_fruits = set(
    outer_splits["fruit_id"]
)

audit_rows = []

for outer_fold in range(OUTER_FOLDS):

    test_fruits = set(
        outer_splits.loc[
            outer_splits["outer_fold"] == outer_fold,
            "fruit_id"
        ]
    )

    train_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner_splits.loc[
            inner_splits["outer_fold"] == outer_fold,
            "fruit_id"
        ]
    )

    audit_rows.append({
        "outer_fold": outer_fold,
        "n_train_fruits": len(train_fruits),
        "n_test_fruits": len(test_fruits),
        "train_test_overlap": len(
            train_fruits & test_fruits
        ),
        "inner_test_overlap": len(
            inner_fruits & test_fruits
        ),
        "inner_matches_train": (
            inner_fruits == train_fruits
        ),
    })

split_audit = pd.DataFrame(audit_rows)

display(split_audit)

assert (
    split_audit["train_test_overlap"] == 0
).all()

assert (
    split_audit["inner_test_overlap"] == 0
).all()

assert split_audit["inner_matches_train"].all()

split_audit.to_csv(
    TABLES / "02_split_audit.csv",
    index=False
)


# 5. Mapa día ↔ sesión y tipo de stress test

En esta campaña cada `storage_day` corresponde a un único `day_code`.

Para el stress test:

- `0`: extrapolación inferior;
- `8`: extrapolación superior;
- días intermedios: interpolación entre otros días observados.


In [ ]:
session_map_rows = []

for camera, df in spectral_data.items():

    tmp = (
        df.groupby("storage_days_model")
        .agg(
            n_day_codes=("day_code", "nunique"),
            day_code=("day_code", "first"),
            n_fruits=("fruit_id", "nunique"),
        )
        .reset_index()
    )

    assert (
        tmp["n_day_codes"] == 1
    ).all()

    for _, row in tmp.iterrows():

        d = int(
            row["storage_days_model"]
        )

        if d == 0:
            mode = "extrapolation_low"
        elif d == 8:
            mode = "extrapolation_high"
        else:
            mode = "interpolation"

        session_map_rows.append({
            "camera_type": camera,
            "storage_days": d,
            "day_code": row["day_code"],
            "n_fruits": int(row["n_fruits"]),
            "stress_test_mode": mode,
        })

session_map = pd.DataFrame(
    session_map_rows
)

display(
    session_map[
        session_map["camera_type"] == "VIS_COR"
    ]
)

session_map.to_csv(
    TABLES / "03_day_session_map.csv",
    index=False
)


# 6. Funciones de preprocesamiento y métricas


In [ ]:
def preprocess_spectra(
    X,
    wavelength_grid,
    method
):

    X = np.asarray(
        X,
        dtype=float
    )

    if method == "raw":
        return X.copy()

    if method == "snv":

        mu = np.mean(
            X,
            axis=1,
            keepdims=True
        )

        sd = np.std(
            X,
            axis=1,
            keepdims=True
        )

        sd[
            (~np.isfinite(sd))
            |
            (sd == 0)
        ] = 1.0

        return (
            X - mu
        ) / sd

    if method == "sg1":

        grid = np.asarray(
            wavelength_grid,
            dtype=float
        )

        delta = float(
            np.median(
                np.diff(grid)
            )
        )

        return savgol_filter(
            X,
            window_length=11,
            polyorder=2,
            deriv=1,
            delta=delta,
            axis=1,
            mode="interp"
        )

    raise ValueError(
        f"Unknown preprocessing: {method}"
    )


def fruit_balanced_mae(
    prediction_df
):

    tmp = prediction_df.copy()

    tmp["abs_error"] = np.abs(
        tmp["y_true"]
        -
        tmp["y_pred"]
    )

    return float(
        tmp.groupby(
            "fruit_id"
        )["abs_error"]
        .mean()
        .mean()
    )


def day_balanced_mae(
    prediction_df
):

    tmp = prediction_df.copy()

    tmp["abs_error"] = np.abs(
        tmp["y_true"]
        -
        tmp["y_pred"]
    )

    return float(
        tmp.groupby(
            "y_true"
        )["abs_error"]
        .mean()
        .mean()
    )


def metric_bundle(
    prediction_df
):

    y_true = prediction_df[
        "y_true"
    ].to_numpy(dtype=float)

    y_pred = prediction_df[
        "y_pred"
    ].to_numpy(dtype=float)

    residual = (
        y_pred - y_true
    )

    rho, _ = spearmanr(
        y_true,
        y_pred
    )

    return {
        "n_observations":
            int(len(prediction_df)),

        "n_fruits":
            int(
                prediction_df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y_true,
                    y_pred
                )
            ),

        "spearman_rho":
            float(rho),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <= 1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                prediction_df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                prediction_df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


def fit_ridge(
    X_train,
    y_train,
    X_test,
    alpha,
    sample_weight=None
):

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(
        X_train
    )

    X_test_s = scaler.transform(
        X_test
    )

    model = Ridge(
        alpha=float(alpha)
    )

    model.fit(
        X_train_s,
        y_train,
        sample_weight=sample_weight
    )

    pred = model.predict(
        X_test_s
    )

    return np.asarray(
        pred,
        dtype=float
    )


def parse_params_json(
    text
):

    if isinstance(text, dict):
        return text

    return json.loads(
        str(text)
    )


# 7. Grid Ridge para session-excluded nested CV

La selección interna repite exactamente la lógica del stress test:

Para validar el día \(d\), se entrenan los frutos internos de training **excluyendo todas las filas con día \(d\)**.


In [ ]:
SESSION_PREPROCESSING = [
    "raw",
    "snv",
    "sg1",
]

SESSION_ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

print(
    "Candidate configs:",
    len(SESSION_PREPROCESSING)
    *
    len(SESSION_ALPHAS)
)


## 8. Selección interna session-excluded


In [ ]:
def inner_select_session_excluded_ridge(
    camera,
    outer_fold,
    development_df,
    wavelength_grid,
    inner_assignment
):

    wl_cols = [
        c for c in development_df.columns
        if c.startswith("wl_")
    ]

    search_rows = []

    best = None

    configs = list(
        itertools.product(
            SESSION_PREPROCESSING,
            SESSION_ALPHAS
        )
    )

    iterator = tqdm(
        configs,
        desc=(
            f"{camera} | outer {outer_fold} | "
            "session-excluded inner CV"
        ),
        leave=False
    )

    for preprocessing, alpha in iterator:

        pred_parts = []
        failed = False
        failure = None

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment["inner_fold"]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            inner_train_df = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            inner_val_df = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            assert (
                set(
                    inner_train_df[
                        "fruit_id"
                    ]
                )
                &
                set(
                    inner_val_df[
                        "fruit_id"
                    ]
                )
                ==
                set()
            )

            for target_day in sorted(
                inner_val_df[
                    "storage_days_model"
                ].unique()
            ):

                val_day = inner_val_df[
                    inner_val_df[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ].copy()

                # Prueba de estrés:
                # quitar la sesión/día objetivo del entrenamiento.
                train_excluded = inner_train_df[
                    inner_train_df[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ].copy()

                if len(
                    train_excluded[
                        "storage_days_model"
                    ].unique()
                ) < 2:

                    failed = True
                    failure = (
                        "Too few target days after exclusion."
                    )
                    break

                try:

                    X_train = preprocess_spectra(
                        train_excluded[
                            wl_cols
                        ].to_numpy(
                            dtype=float
                        ),
                        wavelength_grid,
                        preprocessing
                    )

                    X_val = preprocess_spectra(
                        val_day[
                            wl_cols
                        ].to_numpy(
                            dtype=float
                        ),
                        wavelength_grid,
                        preprocessing
                    )

                    y_train = train_excluded[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    )

                    pred = fit_ridge(
                        X_train,
                        y_train,
                        X_val,
                        alpha=alpha,
                        sample_weight=None
                    )

                    pred_parts.append(
                        pd.DataFrame({
                            "fruit_id":
                                val_day[
                                    "fruit_id"
                                ].to_numpy(
                                    dtype=int
                                ),

                            "y_true":
                                val_day[
                                    "storage_days_model"
                                ].to_numpy(
                                    dtype=float
                                ),

                            "y_pred":
                                pred,

                            "inner_fold":
                                int(
                                    inner_fold
                                ),

                            "target_day":
                                int(
                                    target_day
                                ),
                        })
                    )

                except Exception as exc:

                    failed = True
                    failure = repr(exc)
                    break

            if failed:
                break

        if failed:

            search_rows.append({
                "camera_type": camera,
                "outer_fold": int(outer_fold),
                "preprocessing": preprocessing,
                "alpha": float(alpha),
                "inner_fruit_balanced_mae": np.inf,
                "inner_day_balanced_mae": np.inf,
                "inner_mae": np.inf,
                "failed": True,
                "failure": failure,
            })

            continue

        inner_pred = pd.concat(
            pred_parts,
            ignore_index=True
        )

        assert (
            inner_pred[
                ["fruit_id", "y_true"]
            ]
            .duplicated()
            .sum()
            ==
            0
        )

        score_fb = fruit_balanced_mae(
            inner_pred
        )

        score_db = day_balanced_mae(
            inner_pred
        )

        score_mae = float(
            mean_absolute_error(
                inner_pred["y_true"],
                inner_pred["y_pred"]
            )
        )

        search_rows.append({
            "camera_type": camera,
            "outer_fold": int(outer_fold),
            "preprocessing": preprocessing,
            "alpha": float(alpha),
            "inner_fruit_balanced_mae": score_fb,
            "inner_day_balanced_mae": score_db,
            "inner_mae": score_mae,
            "failed": False,
            "failure": None,
        })

        rank = (
            score_fb,
            score_db,
            score_mae,
            preprocessing,
            float(alpha),
        )

        if (
            best is None
            or
            rank < best["rank"]
        ):

            best = {
                "rank": rank,
                "preprocessing": preprocessing,
                "alpha": float(alpha),
                "inner_fruit_balanced_mae": score_fb,
                "inner_day_balanced_mae": score_db,
                "inner_mae": score_mae,
            }

    if best is None:
        raise RuntimeError(
            f"No valid session-excluded config: "
            f"{camera}, outer {outer_fold}"
        )

    return (
        best,
        pd.DataFrame(
            search_rows
        )
    )


# 9. Outer OOF session-excluded Ridge

Para cada outer fold y cada día presente en los 8 frutos test:

1. se excluyen los 8 frutos test;
2. se eliminan del desarrollo todas las observaciones del día objetivo;
3. se entrena Ridge con la configuración elegida únicamente en la CV interna session-excluded;
4. se predicen los frutos test de ese día.


In [ ]:
def session_checkpoint_paths(
    camera,
    outer_fold
):

    stem = (
        f"{RUN_SIGNATURE}__"
        f"{camera}__outer{outer_fold}"
    )

    return {
        "pred":
            CHECKPOINT_DIR /
            f"{stem}__session_excluded_pred.csv",

        "selected":
            CHECKPOINT_DIR /
            f"{stem}__session_excluded_selected.json",

        "search":
            CHECKPOINT_DIR /
            f"{stem}__session_excluded_inner_search.csv",
    }


session_pred_parts = []
session_selected_rows = []
session_search_parts = []

all_fruits = set(
    outer_splits["fruit_id"].astype(int)
)

for camera in [
    "VIS",
    "VIS_COR"
]:

    df_camera = spectral_data[
        camera
    ].copy()

    wl_cols = [
        c for c in df_camera.columns
        if c.startswith("wl_")
    ]

    grid = wavelengths[
        camera
    ]

    for outer_fold in range(
        OUTER_FOLDS
    ):

        paths = session_checkpoint_paths(
            camera,
            outer_fold
        )

        if (
            RESUME
            and
            paths["pred"].exists()
            and
            paths["selected"].exists()
        ):

            session_pred_parts.append(
                pd.read_csv(
                    paths["pred"]
                )
            )

            session_selected_rows.append(
                json.loads(
                    paths["selected"].read_text(
                        encoding="utf-8"
                    )
                )
            )

            if paths["search"].exists():
                session_search_parts.append(
                    pd.read_csv(
                        paths["search"]
                    )
                )

            print(
                f"↪ checkpoint: {camera} outer {outer_fold}"
            )

            continue

        test_fruits = set(
            outer_splits.loc[
                outer_splits["outer_fold"]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            all_fruits
            -
            test_fruits
        )

        development_df = df_camera[
            df_camera[
                "fruit_id"
            ].isin(
                development_fruits
            )
        ].copy()

        test_df = df_camera[
            df_camera[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        inner_assignment = (
            inner_splits[
                inner_splits[
                    "outer_fold"
                ]
                ==
                outer_fold
            ][
                [
                    "fruit_id",
                    "inner_fold"
                ]
            ]
            .copy()
        )

        best, search_df = (
            inner_select_session_excluded_ridge(
                camera=camera,
                outer_fold=outer_fold,
                development_df=development_df,
                wavelength_grid=grid,
                inner_assignment=inner_assignment
            )
        )

        pred_days = []

        for target_day in sorted(
            test_df[
                "storage_days_model"
            ].unique()
        ):

            test_day = test_df[
                test_df[
                    "storage_days_model"
                ]
                ==
                target_day
            ].copy()

            train_excluded = development_df[
                development_df[
                    "storage_days_model"
                ]
                !=
                target_day
            ].copy()

            assert (
                target_day
                not in set(
                    train_excluded[
                        "storage_days_model"
                    ]
                )
            )

            X_train = preprocess_spectra(
                train_excluded[
                    wl_cols
                ].to_numpy(
                    dtype=float
                ),
                grid,
                best[
                    "preprocessing"
                ]
            )

            X_test = preprocess_spectra(
                test_day[
                    wl_cols
                ].to_numpy(
                    dtype=float
                ),
                grid,
                best[
                    "preprocessing"
                ]
            )

            y_train = train_excluded[
                "storage_days_model"
            ].to_numpy(
                dtype=float
            )

            pred = fit_ridge(
                X_train,
                y_train,
                X_test,
                alpha=best[
                    "alpha"
                ]
            )

            mode = (
                "extrapolation_low"
                if int(target_day) == 0
                else (
                    "extrapolation_high"
                    if int(target_day) == 8
                    else "interpolation"
                )
            )

            pred_days.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "model":
                        "Ridge_session_excluded",

                    "fruit_id":
                        test_day[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        test_day[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "day_code":
                        test_day[
                            "day_code"
                        ].astype(
                            str
                        ).to_numpy(),

                    "stress_test_mode":
                        mode,

                    "preprocessing":
                        best[
                            "preprocessing"
                        ],

                    "alpha":
                        float(
                            best[
                                "alpha"
                            ]
                        ),

                    "y_true":
                        test_day[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        pred_df = pd.concat(
            pred_days,
            ignore_index=True
        )

        assert (
            pred_df[
                ["fruit_id", "storage_days"]
            ]
            .duplicated()
            .sum()
            ==
            0
        )

        assert (
            set(
                pred_df[
                    "fruit_id"
                ].astype(int)
            )
            ==
            test_fruits
        )

        selected = {
            "camera_type":
                camera,

            "outer_fold":
                int(
                    outer_fold
                ),

            "preprocessing":
                best[
                    "preprocessing"
                ],

            "alpha":
                float(
                    best[
                        "alpha"
                    ]
                ),

            "inner_fruit_balanced_mae":
                float(
                    best[
                        "inner_fruit_balanced_mae"
                    ]
                ),

            "inner_day_balanced_mae":
                float(
                    best[
                        "inner_day_balanced_mae"
                    ]
                ),

            "inner_mae":
                float(
                    best[
                        "inner_mae"
                    ]
                ),
        }

        pred_df.to_csv(
            paths["pred"],
            index=False
        )

        search_df.to_csv(
            paths["search"],
            index=False
        )

        paths["selected"].write_text(
            json.dumps(
                selected,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        session_pred_parts.append(
            pred_df
        )

        session_selected_rows.append(
            selected
        )

        session_search_parts.append(
            search_df
        )

session_oof = pd.concat(
    session_pred_parts,
    ignore_index=True
)

session_selected = pd.DataFrame(
    session_selected_rows
)

session_inner_search = pd.concat(
    session_search_parts,
    ignore_index=True
)

for camera in ["VIS", "VIS_COR"]:

    g = session_oof[
        session_oof[
            "camera_type"
        ]
        ==
        camera
    ]

    assert len(g) == 199
    assert g["fruit_id"].nunique() == 40

print("✅ Session-excluded OOF completo.")


## 10. Métricas session-excluded


In [ ]:
session_metric_rows = []

for camera, g in session_oof.groupby(
    "camera_type"
):

    metrics = metric_bundle(
        g
    )

    metrics["camera_type"] = camera
    metrics["model"] = "Ridge_session_excluded"

    session_metric_rows.append(
        metrics
    )

session_metrics = pd.DataFrame(
    session_metric_rows
)

display(
    session_metrics
)

session_metrics.to_csv(
    TABLES /
    "04_session_excluded_oof_metrics.csv",
    index=False
)

session_selected.to_csv(
    TABLES /
    "05_session_excluded_selected_configs.csv",
    index=False
)

session_inner_search.to_csv(
    TABLES /
    "06_session_excluded_inner_search.csv",
    index=False
)


# 11. Baseline extremadamente simple: intensidad espectral global

Para cada fruto-día se calcula:

\[
I = 
rac{1}{p}\sum_{\lambda=1}^{p}R(\lambda)
\]

y se ajusta una regresión lineal:

\[
storage\_days = eta_0 + eta_1 I
\]

Se evalúan dos versiones:

- `GlobalMean_standard`: fruit-disjoint, usando todos los días del desarrollo;
- `GlobalMean_session_excluded`: para predecir día \(d\), elimina del training el día \(d\).

Este análisis mide cuánto desempeño puede obtenerse con **una sola variable de amplitud global**.


In [ ]:
global_mean_pred_parts = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    df_camera = spectral_data[
        camera
    ].copy()

    wl_cols = [
        c for c in df_camera.columns
        if c.startswith("wl_")
    ]

    df_camera[
        "global_mean_intensity"
    ] = (
        df_camera[
            wl_cols
        ]
        .mean(
            axis=1
        )
    )

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer_splits.loc[
                outer_splits[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_df = df_camera[
            ~df_camera[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        test_df = df_camera[
            df_camera[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        # --------------------------------------------
        # Standard fruit-disjoint global mean
        # --------------------------------------------

        model = LinearRegression()

        model.fit(
            development_df[
                [
                    "global_mean_intensity"
                ]
            ],
            development_df[
                "storage_days_model"
            ]
        )

        pred = model.predict(
            test_df[
                [
                    "global_mean_intensity"
                ]
            ]
        )

        global_mean_pred_parts.append(
            pd.DataFrame({
                "camera_type":
                    camera,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "model":
                    "GlobalMean_standard",

                "fruit_id":
                    test_df[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    test_df[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    test_df[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    pred,
            })
        )

        # --------------------------------------------
        # Session-excluded global mean
        # --------------------------------------------

        for target_day in sorted(
            test_df[
                "storage_days_model"
            ].unique()
        ):

            test_day = test_df[
                test_df[
                    "storage_days_model"
                ]
                ==
                target_day
            ].copy()

            train_excluded = development_df[
                development_df[
                    "storage_days_model"
                ]
                !=
                target_day
            ].copy()

            model = LinearRegression()

            model.fit(
                train_excluded[
                    [
                        "global_mean_intensity"
                    ]
                ],
                train_excluded[
                    "storage_days_model"
                ]
            )

            pred = model.predict(
                test_day[
                    [
                        "global_mean_intensity"
                    ]
                ]
            )

            global_mean_pred_parts.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "model":
                        "GlobalMean_session_excluded",

                    "fruit_id":
                        test_day[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        test_day[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        test_day[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

global_mean_oof = pd.concat(
    global_mean_pred_parts,
    ignore_index=True
)

for (
    camera,
    model_name
), g in global_mean_oof.groupby(
    [
        "camera_type",
        "model"
    ]
):

    assert len(g) == 199
    assert g["fruit_id"].nunique() == 40

print("✅ Global-mean OOF completo.")


## 12. Métricas baseline de intensidad global


In [ ]:
global_metric_rows = []

for (
    camera,
    model_name
), g in global_mean_oof.groupby(
    [
        "camera_type",
        "model"
    ]
):

    metrics = metric_bundle(
        g
    )

    metrics.update({
        "camera_type": camera,
        "model": model_name,
    })

    global_metric_rows.append(
        metrics
    )

global_metrics = pd.DataFrame(
    global_metric_rows
)

display(
    global_metrics.sort_values(
        [
            "camera_type",
            "model"
        ]
    )
)

global_metrics.to_csv(
    TABLES /
    "07_global_mean_baseline_metrics.csv",
    index=False
)


# 13. Sensibilidad a ponderación temporal

Se reutiliza la configuración Ridge elegida correctamente dentro de cada outer fold en el notebook 02.

No se vuelve a elegir hiperparámetros mirando el test.

Se prueban:

- `Ridge_day_weighted`: peso inverso a la frecuencia del día en training;
- `Ridge_fruit_day_weighted`: producto de peso inverso por fruto y por día.

Los pesos se normalizan para tener media 1.


In [ ]:
def normalized_weights(
    train_df,
    mode
):

    if mode == "day":

        day_counts = (
            train_df[
                "storage_days_model"
            ]
            .value_counts()
            .to_dict()
        )

        w = np.array(
            [
                1.0
                /
                day_counts[
                    int(d)
                ]
                for d in train_df[
                    "storage_days_model"
                ]
            ],
            dtype=float
        )

    elif mode == "fruit_day":

        day_counts = (
            train_df[
                "storage_days_model"
            ]
            .value_counts()
            .to_dict()
        )

        fruit_counts = (
            train_df[
                "fruit_id"
            ]
            .value_counts()
            .to_dict()
        )

        w = np.array(
            [
                (
                    1.0
                    /
                    day_counts[
                        int(d)
                    ]
                )
                *
                (
                    1.0
                    /
                    fruit_counts[
                        int(f)
                    ]
                )
                for f, d in zip(
                    train_df[
                        "fruit_id"
                    ],
                    train_df[
                        "storage_days_model"
                    ]
                )
            ],
            dtype=float
        )

    else:

        raise ValueError(
            mode
        )

    w = (
        w
        /
        np.mean(
            w
        )
    )

    return w


weighted_pred_parts = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    df_camera = spectral_data[
        camera
    ].copy()

    wl_cols = [
        c for c in df_camera.columns
        if c.startswith("wl_")
    ]

    grid = wavelengths[
        camera
    ]

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer_splits.loc[
                outer_splits[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        train_df = df_camera[
            ~df_camera[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        test_df = df_camera[
            df_camera[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        config_row = selected_configs[
            (
                selected_configs[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                selected_configs[
                    "outer_fold"
                ]
                ==
                outer_fold
            )
            &
            (
                selected_configs[
                    "model"
                ]
                ==
                "Ridge"
            )
        ]

        if len(config_row) != 1:
            raise RuntimeError(
                f"No unique Ridge config for "
                f"{camera}, outer {outer_fold}"
            )

        config_row = config_row.iloc[0]

        preprocessing = str(
            config_row[
                "preprocessing"
            ]
        )

        params = parse_params_json(
            config_row[
                "params_json"
            ]
        )

        alpha = float(
            params[
                "alpha"
            ]
        )

        X_train = preprocess_spectra(
            train_df[
                wl_cols
            ].to_numpy(
                dtype=float
            ),
            grid,
            preprocessing
        )

        X_test = preprocess_spectra(
            test_df[
                wl_cols
            ].to_numpy(
                dtype=float
            ),
            grid,
            preprocessing
        )

        y_train = train_df[
            "storage_days_model"
        ].to_numpy(
            dtype=float
        )

        for weight_mode in [
            "day",
            "fruit_day"
        ]:

            weights = normalized_weights(
                train_df,
                mode=weight_mode
            )

            pred = fit_ridge(
                X_train,
                y_train,
                X_test,
                alpha=alpha,
                sample_weight=weights
            )

            weighted_pred_parts.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "model":
                        (
                            "Ridge_day_weighted"
                            if weight_mode == "day"
                            else "Ridge_fruit_day_weighted"
                        ),

                    "fruit_id":
                        test_df[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        test_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "preprocessing":
                        preprocessing,

                    "alpha":
                        alpha,

                    "y_true":
                        test_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

weighted_oof = pd.concat(
    weighted_pred_parts,
    ignore_index=True
)

for (
    camera,
    model_name
), g in weighted_oof.groupby(
    [
        "camera_type",
        "model"
    ]
):

    assert len(g) == 199
    assert g["fruit_id"].nunique() == 40

print("✅ Sensibilidad de ponderación terminada.")


## 14. Métricas de modelos ponderados


In [ ]:
weighted_metric_rows = []

for (
    camera,
    model_name
), g in weighted_oof.groupby(
    [
        "camera_type",
        "model"
    ]
):

    metrics = metric_bundle(
        g
    )

    metrics.update({
        "camera_type": camera,
        "model": model_name,
    })

    weighted_metric_rows.append(
        metrics
    )

weighted_metrics = pd.DataFrame(
    weighted_metric_rows
)

display(
    weighted_metrics.sort_values(
        [
            "camera_type",
            "fruit_balanced_mae"
        ]
    )
)

weighted_metrics.to_csv(
    TABLES /
    "08_weighted_ridge_metrics.csv",
    index=False
)


# 15. Construir tabla maestra de OOF para robustez

Se combinan:

- Ridge estándar del notebook 02;
- PLSR estándar del notebook 02;
- session-excluded Ridge;
- global mean standard;
- global mean session-excluded;
- Ridge ponderado por día;
- Ridge ponderado fruto×día.


In [ ]:
standard_subset = baseline_oof[
    baseline_oof[
        "model"
    ].isin(
        [
            "Ridge",
            "PLSR",
        ]
    )
].copy()

standard_subset["model"] = (
    standard_subset[
        "model"
    ]
    .map({
        "Ridge":
            "Ridge_standard",

        "PLSR":
            "PLSR_standard",
    })
)

keep_cols = [
    "camera_type",
    "outer_fold",
    "model",
    "fruit_id",
    "storage_days",
    "y_true",
    "y_pred",
]

robustness_oof = pd.concat(
    [
        standard_subset[
            keep_cols
        ],
        session_oof[
            keep_cols
        ],
        global_mean_oof[
            keep_cols
        ],
        weighted_oof[
            keep_cols
        ],
    ],
    ignore_index=True
)

robustness_oof["abs_error"] = np.abs(
    robustness_oof[
        "y_true"
    ]
    -
    robustness_oof[
        "y_pred"
    ]
)

robustness_oof["residual"] = (
    robustness_oof[
        "y_pred"
    ]
    -
    robustness_oof[
        "y_true"
    ]
)

display(
    robustness_oof[
        [
            "camera_type",
            "model"
        ]
    ]
    .value_counts()
    .sort_index()
    .rename("n")
    .to_frame()
)


## 16. Métricas maestras de robustez


In [ ]:
robustness_metric_rows = []

for (
    camera,
    model_name
), g in robustness_oof.groupby(
    [
        "camera_type",
        "model"
    ]
):

    metrics = metric_bundle(
        g
    )

    metrics.update({
        "camera_type": camera,
        "model": model_name,
    })

    robustness_metric_rows.append(
        metrics
    )

robustness_metrics = (
    pd.DataFrame(
        robustness_metric_rows
    )
    .sort_values(
        [
            "camera_type",
            "fruit_balanced_mae"
        ]
    )
    .reset_index(drop=True)
)

display(
    robustness_metrics
)

robustness_metrics.to_csv(
    TABLES /
    "09_robustness_all_model_metrics.csv",
    index=False
)


# 17. Retención de desempeño al excluir la sesión objetivo

Definimos:

\[
	ext{MAE retention ratio}
=

rac{MAE_{session\ excluded}}
{MAE_{standard}}
\]

Valores cercanos a 1 indican poca degradación; valores altos indican que el stress test es mucho más difícil.

Esta razón no separa por sí sola fisiología y sesión.


In [ ]:
retention_rows = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    standard = robustness_metrics[
        (
            robustness_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            robustness_metrics[
                "model"
            ]
            ==
            "Ridge_standard"
        )
    ].iloc[0]

    excluded = robustness_metrics[
        (
            robustness_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            robustness_metrics[
                "model"
            ]
            ==
            "Ridge_session_excluded"
        )
    ].iloc[0]

    retention_rows.append({
        "camera_type":
            camera,

        "standard_fruit_balanced_mae":
            float(
                standard[
                    "fruit_balanced_mae"
                ]
            ),

        "session_excluded_fruit_balanced_mae":
            float(
                excluded[
                    "fruit_balanced_mae"
                ]
            ),

        "mae_retention_ratio":
            float(
                excluded[
                    "fruit_balanced_mae"
                ]
                /
                standard[
                    "fruit_balanced_mae"
                ]
            ),

        "delta_mae_days":
            float(
                excluded[
                    "fruit_balanced_mae"
                ]
                -
                standard[
                    "fruit_balanced_mae"
                ]
            ),

        "standard_r2":
            float(
                standard[
                    "r2"
                ]
            ),

        "session_excluded_r2":
            float(
                excluded[
                    "r2"
                ]
            ),

        "delta_r2":
            float(
                excluded[
                    "r2"
                ]
                -
                standard[
                    "r2"
                ]
            ),
    })

retention = pd.DataFrame(
    retention_rows
)

display(
    retention
)

retention.to_csv(
    TABLES /
    "10_session_exclusion_retention.csv",
    index=False
)


# 18. Error y sesgo por día

Se presta atención especial a:

- día 7;
- día 8;
- comparación estándar vs ponderado;
- comparación estándar vs session-excluded.


In [ ]:
day_metric_rows = []

for (
    camera,
    model_name,
    day
), g in robustness_oof.groupby(
    [
        "camera_type",
        "model",
        "storage_days"
    ]
):

    y_true = g[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    y_pred = g[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    error = (
        y_pred - y_true
    )

    day_metric_rows.append({
        "camera_type":
            camera,

        "model":
            model_name,

        "storage_days":
            int(day),

        "n":
            int(len(g)),

        "mae":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "median_ae":
            float(
                np.median(
                    np.abs(
                        error
                    )
                )
            ),

        "bias":
            float(
                np.mean(
                    error
                )
            ),

        "pred_mean":
            float(
                np.mean(
                    y_pred
                )
            ),
    })

day_metrics = pd.DataFrame(
    day_metric_rows
)

display(
    day_metrics[
        (
            day_metrics[
                "camera_type"
            ]
            ==
            "VIS_COR"
        )
        &
        (
            day_metrics[
                "model"
            ].isin(
                [
                    "Ridge_standard",
                    "Ridge_session_excluded",
                    "Ridge_day_weighted",
                    "Ridge_fruit_day_weighted",
                ]
            )
        )
    ]
    .sort_values(
        [
            "model",
            "storage_days"
        ]
    )
)

day_metrics.to_csv(
    TABLES /
    "11_error_bias_by_day.csv",
    index=False
)


## 19. Figura — error por día VIS_COR


In [ ]:
models_for_day_plot = [
    "Ridge_standard",
    "Ridge_session_excluded",
    "Ridge_day_weighted",
    "Ridge_fruit_day_weighted",
    "GlobalMean_standard",
]

fig, ax = plt.subplots(
    figsize=(10, 5.5)
)

for model_name in models_for_day_plot:

    g = day_metrics[
        (
            day_metrics[
                "camera_type"
            ]
            ==
            "VIS_COR"
        )
        &
        (
            day_metrics[
                "model"
            ]
            ==
            model_name
        )
    ].sort_values(
        "storage_days"
    )

    ax.plot(
        g[
            "storage_days"
        ],
        g[
            "mae"
        ],
        marker="o",
        label=model_name
    )

ax.set_xlabel(
    "Días de almacenamiento"
)

ax.set_ylabel(
    "MAE (días)"
)

ax.set_title(
    "VIS_COR — error por día bajo pruebas de robustez"
)

ax.legend(
    fontsize=8
)

ax.grid(
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "01_viscor_mae_by_day_robustness.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 20. Figura — bias por día VIS_COR


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 5.5)
)

for model_name in [
    "Ridge_standard",
    "Ridge_day_weighted",
    "Ridge_fruit_day_weighted",
    "Ridge_session_excluded",
]:

    g = day_metrics[
        (
            day_metrics[
                "camera_type"
            ]
            ==
            "VIS_COR"
        )
        &
        (
            day_metrics[
                "model"
            ]
            ==
            model_name
        )
    ].sort_values(
        "storage_days"
    )

    ax.plot(
        g[
            "storage_days"
        ],
        g[
            "bias"
        ],
        marker="o",
        label=model_name
    )

ax.axhline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Días de almacenamiento"
)

ax.set_ylabel(
    "Bias medio = predicho − real (días)"
)

ax.set_title(
    "VIS_COR — sesgo temporal"
)

ax.legend(
    fontsize=8
)

ax.grid(
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "02_viscor_bias_by_day.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 21. Interpolación vs extrapolación del stress test

Se resumen por separado:

- días internos 1,2,3,4,6,7;
- día 0;
- día 8.

Esto evita interpretar los extremos como si fueran equivalentes a los días internos.


In [ ]:
session_oof_mode = session_oof.copy()

session_oof_mode[
    "abs_error"
] = np.abs(
    session_oof_mode[
        "y_true"
    ]
    -
    session_oof_mode[
        "y_pred"
    ]
)

mode_summary = (
    session_oof_mode.groupby(
        [
            "camera_type",
            "stress_test_mode"
        ]
    )
    .agg(
        n=("abs_error", "size"),
        n_fruits=("fruit_id", "nunique"),
        mae=("abs_error", "mean"),
        median_ae=("abs_error", "median"),
    )
    .reset_index()
)

display(
    mode_summary
)

mode_summary.to_csv(
    TABLES /
    "12_session_stress_interpolation_extrapolation.csv",
    index=False
)


# 22. Comparaciones pareadas por fruto

La unidad de remuestreo es el **fruto físico**.

La estadística es:

\[
\Delta =
MAE_A - MAE_B
\]

- \(\Delta < 0\): A mejor.
- \(\Delta > 0\): B mejor.

Los intervalos bootstrap son exploratorios.


In [ ]:
def per_fruit_mae_table(
    prediction_df,
    camera,
    model_name
):

    g = prediction_df[
        (
            prediction_df[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            prediction_df[
                "model"
            ]
            ==
            model_name
        )
    ].copy()

    g[
        "abs_error"
    ] = np.abs(
        g[
            "y_true"
        ]
        -
        g[
            "y_pred"
        ]
    )

    return (
        g.groupby(
            "fruit_id"
        )["abs_error"]
        .mean()
        .rename(
            "fruit_mae"
        )
    )


def paired_fruit_bootstrap(
    table_a,
    table_b,
    label_a,
    label_b,
    n_boot=5000,
    seed=20260818
):

    common = sorted(
        set(
            table_a.index
        )
        &
        set(
            table_b.index
        )
    )

    a = table_a.loc[
        common
    ].to_numpy(
        dtype=float
    )

    b = table_b.loc[
        common
    ].to_numpy(
        dtype=float
    )

    diff = (
        a - b
    )

    rng_local = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    n = len(
        diff
    )

    for i in range(
        n_boot
    ):

        idx = rng_local.integers(
            0,
            n,
            size=n
        )

        boot[i] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            f"{label_a} minus {label_b}",

        "n_fruits":
            int(n),

        "mean_delta_mae_days":
            float(
                np.mean(
                    diff
                )
            ),

        "median_delta_mae_days":
            float(
                np.median(
                    diff
                )
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_a_better":
            float(
                np.mean(
                    diff < 0
                )
            ),
    }


In [ ]:
paired_rows = []

# VIS_COR vs VIS, Ridge estándar
a = per_fruit_mae_table(
    robustness_oof,
    "VIS_COR",
    "Ridge_standard"
)

b = per_fruit_mae_table(
    robustness_oof,
    "VIS",
    "Ridge_standard"
)

paired_rows.append(
    paired_fruit_bootstrap(
        a,
        b,
        "VIS_COR Ridge standard",
        "VIS Ridge standard",
        n_boot=BOOTSTRAP_REPS,
        seed=RANDOM_SEED + 1
    )
)

# Ridge vs PLSR en VIS_COR
a = per_fruit_mae_table(
    robustness_oof,
    "VIS_COR",
    "Ridge_standard"
)

b = per_fruit_mae_table(
    robustness_oof,
    "VIS_COR",
    "PLSR_standard"
)

paired_rows.append(
    paired_fruit_bootstrap(
        a,
        b,
        "VIS_COR Ridge standard",
        "VIS_COR PLSR standard",
        n_boot=BOOTSTRAP_REPS,
        seed=RANDOM_SEED + 2
    )
)

# Ridge estándar vs session-excluded
for camera, seed_offset in [
    ("VIS", 10),
    ("VIS_COR", 20),
]:

    a = per_fruit_mae_table(
        robustness_oof,
        camera,
        "Ridge_standard"
    )

    b = per_fruit_mae_table(
        robustness_oof,
        camera,
        "Ridge_session_excluded"
    )

    paired_rows.append(
        paired_fruit_bootstrap(
            a,
            b,
            f"{camera} Ridge standard",
            f"{camera} Ridge session-excluded",
            n_boot=BOOTSTRAP_REPS,
            seed=RANDOM_SEED + seed_offset
        )
    )

# Ridge estándar vs ponderaciones VIS_COR
for model_name, seed_offset in [
    ("Ridge_day_weighted", 30),
    ("Ridge_fruit_day_weighted", 40),
]:

    a = per_fruit_mae_table(
        robustness_oof,
        "VIS_COR",
        "Ridge_standard"
    )

    b = per_fruit_mae_table(
        robustness_oof,
        "VIS_COR",
        model_name
    )

    paired_rows.append(
        paired_fruit_bootstrap(
            a,
            b,
            "VIS_COR Ridge standard",
            f"VIS_COR {model_name}",
            n_boot=BOOTSTRAP_REPS,
            seed=RANDOM_SEED + seed_offset
        )
    )

paired_results = pd.DataFrame(
    paired_rows
)

display(
    paired_results
)

paired_results.to_csv(
    TABLES /
    "13_paired_fruit_bootstrap_comparisons.csv",
    index=False
)


# 23. Cuánto explica la intensidad global

Se compara el baseline de una sola variable con Ridge multibanda.

Una diferencia grande a favor de Ridge indica que la **forma espectral multibanda** aporta información adicional respecto a una simple amplitud global.


In [ ]:
intensity_comparison_rows = []

for camera in [
    "VIS",
    "VIS_COR"
]:

    ridge = robustness_metrics[
        (
            robustness_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            robustness_metrics[
                "model"
            ]
            ==
            "Ridge_standard"
        )
    ].iloc[0]

    global_standard = robustness_metrics[
        (
            robustness_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            robustness_metrics[
                "model"
            ]
            ==
            "GlobalMean_standard"
        )
    ].iloc[0]

    global_excluded = robustness_metrics[
        (
            robustness_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            robustness_metrics[
                "model"
            ]
            ==
            "GlobalMean_session_excluded"
        )
    ].iloc[0]

    intensity_comparison_rows.append({
        "camera_type":
            camera,

        "ridge_standard_fruit_balanced_mae":
            float(
                ridge[
                    "fruit_balanced_mae"
                ]
            ),

        "global_mean_standard_fruit_balanced_mae":
            float(
                global_standard[
                    "fruit_balanced_mae"
                ]
            ),

        "ridge_advantage_over_global_mean_days":
            float(
                global_standard[
                    "fruit_balanced_mae"
                ]
                -
                ridge[
                    "fruit_balanced_mae"
                ]
            ),

        "global_mean_session_excluded_fruit_balanced_mae":
            float(
                global_excluded[
                    "fruit_balanced_mae"
                ]
            ),
    })

intensity_comparison = pd.DataFrame(
    intensity_comparison_rows
)

display(
    intensity_comparison
)

intensity_comparison.to_csv(
    TABLES /
    "14_global_intensity_vs_multiband.csv",
    index=False
)


## 24. Figura — modelos de robustez por cámara


In [ ]:
models_for_summary = [
    "Ridge_standard",
    "Ridge_session_excluded",
    "Ridge_day_weighted",
    "Ridge_fruit_day_weighted",
    "GlobalMean_standard",
    "GlobalMean_session_excluded",
]

for camera in [
    "VIS",
    "VIS_COR"
]:

    g = robustness_metrics[
        (
            robustness_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            robustness_metrics[
                "model"
            ].isin(
                models_for_summary
            )
        )
    ].copy()

    g["model"] = pd.Categorical(
        g["model"],
        categories=models_for_summary,
        ordered=True
    )

    g = g.sort_values(
        "model"
    )

    fig, ax = plt.subplots(
        figsize=(10, 4.8)
    )

    ax.bar(
        g[
            "model"
        ].astype(str),
        g[
            "fruit_balanced_mae"
        ]
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_xlabel(
        "Modelo / stress test"
    )

    ax.set_title(
        f"{camera} — robustez del baseline"
    )

    ax.tick_params(
        axis="x",
        rotation=35
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"03_robustness_model_comparison_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 25. Predicho vs real — session-excluded VIS_COR

La gráfica debe interpretarse distinguiendo:
- interpolación;
- extrapolación de día 0;
- extrapolación de día 8.


In [ ]:
g = session_oof[
    session_oof[
        "camera_type"
    ]
    ==
    "VIS_COR"
].copy()

fig, ax = plt.subplots(
    figsize=(6.5, 5.8)
)

mode_codes = {
    "interpolation": 0,
    "extrapolation_low": 1,
    "extrapolation_high": 2,
}

c = g[
    "stress_test_mode"
].map(
    mode_codes
).to_numpy()

scatter = ax.scatter(
    g[
        "y_true"
    ],
    g[
        "y_pred"
    ],
    c=c,
    s=38,
    alpha=0.8
)

low = min(
    -0.5,
    float(
        g[
            [
                "y_true",
                "y_pred"
            ]
        ].min().min()
    )
    -
    0.25
)

high = max(
    8.5,
    float(
        g[
            [
                "y_true",
                "y_pred"
            ]
        ].max().max()
    )
    +
    0.25
)

ax.plot(
    [
        low,
        high
    ],
    [
        low,
        high
    ],
    linestyle="--"
)

ax.set_xlim(
    low,
    high
)

ax.set_ylim(
    low,
    high
)

ax.set_xlabel(
    "Storage days reales"
)

ax.set_ylabel(
    "Storage days predichos"
)

ax.set_title(
    "VIS_COR — Ridge session-excluded\n"
    "fruit-disjoint OOF"
)

ax.grid(
    alpha=0.15
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "04_viscor_session_excluded_predicted_vs_true.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 26. Diagnóstico de selección session-excluded

Se revisa si RAW continúa dominando cuando el día/sesión objetivo no existe en training.


In [ ]:
selection_stability = (
    session_selected.groupby(
        [
            "camera_type",
            "preprocessing",
            "alpha"
        ]
    )
    .size()
    .reset_index(
        name="n_outer_folds_selected"
    )
    .sort_values(
        [
            "camera_type",
            "n_outer_folds_selected"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    selection_stability
)

selection_stability.to_csv(
    TABLES /
    "15_session_excluded_selection_stability.csv",
    index=False
)


# 27. Guardar resultados centrales


In [ ]:
SESSION_OOF_PATH = (
    RESULTS /
    "kaki_session_excluded_oof_predictions.csv"
)

ROBUSTNESS_OOF_PATH = (
    RESULTS /
    "kaki_robustness_oof_predictions.csv"
)

ROBUSTNESS_METRICS_PATH = (
    RESULTS /
    "kaki_robustness_metrics.csv"
)

session_oof.to_csv(
    SESSION_OOF_PATH,
    index=False
)

robustness_oof.to_csv(
    ROBUSTNESS_OOF_PATH,
    index=False
)

robustness_metrics.to_csv(
    ROBUSTNESS_METRICS_PATH,
    index=False
)

print("✅ Resultados guardados.")


# 28. Resumen científico automático

El JSON no declara que el stress test haya "eliminado" la confusión de sesión. Solo cuantifica robustez bajo exclusión del día/sesión objetivo.


In [ ]:
def json_safe(obj):

    if isinstance(obj, dict):
        return {
            str(k): json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            json_safe(v)
            for v in obj
        ]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return (
            None
            if np.isnan(obj)
            else float(obj)
        )

    if isinstance(obj, np.bool_):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return json_safe(
            obj.tolist()
        )

    return obj


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "data": {
        "physical_fruits":
            40,

        "fruit_days_per_camera":
            199,

        "storage_days":
            sorted(
                [
                    int(x)
                    for x in spectral_data[
                        "VIS"
                    ][
                        "storage_days_model"
                    ].unique()
                ]
            ),

        "cameras": {
            camera:
                int(
                    len(
                        wavelengths[
                            camera
                        ]
                    )
                )
            for camera in [
                "VIS",
                "VIS_COR"
            ]
        },
    },

    "stress_test_definition": {
        "name":
            "target-session-excluded fruit-disjoint OOF",

        "rule":
            (
                "For each target storage day d, all development rows "
                "from storage_day/session d are excluded before fitting."
            ),

        "inner_selection":
            (
                "Preprocessing and alpha are selected with the same "
                "session-excluded rule inside outer development fruits."
            ),

        "interpretation":
            (
                "This is a robustness stress test, not causal deconfounding. "
                "It also removes the target label d from training."
            ),

        "interpolation_days":
            [
                1,
                2,
                3,
                4,
                6,
                7
            ],

        "extrapolation_low":
            0,

        "extrapolation_high":
            8,
    },

    "standard_vs_session_excluded":
        retention.to_dict(
            orient="records"
        ),

    "global_intensity_comparison":
        intensity_comparison.to_dict(
            orient="records"
        ),

    "paired_fruit_bootstrap":
        paired_results.to_dict(
            orient="records"
        ),

    "limitations": [
        (
            "One experimental campaign cannot causally separate "
            "physiological storage effects from acquisition-session effects."
        ),
        (
            "Session-excluded validation also withholds the target day label, "
            "so performance degradation mixes session robustness with "
            "interpolation/extrapolation difficulty."
        ),
        (
            "Late days contain fewer fruits because of destructive endpoints."
        ),
        (
            "The weighting analyses are sensitivity analyses and do not "
            "constitute a new preregistered primary model."
        ),
        (
            "External batch/campaign validation is still absent."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_robustness_session_bias_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 29. Notas automáticas para revisión


In [ ]:
review_lines = [
    "KAKI — ROBUSTNESS / SESSION-BIAS REVIEW",
    "",
    f"Run signature: {RUN_SIGNATURE}",
    "",
    "This notebook performs stress tests only.",
    "It does NOT prove causal separation of storage time and acquisition session.",
    "",
]

for _, row in retention.iterrows():

    review_lines.extend([
        f"{row['camera_type']}:",
        (
            "  Ridge standard fruit-balanced MAE = "
            f"{row['standard_fruit_balanced_mae']:.4f} d"
        ),
        (
            "  Session-excluded fruit-balanced MAE = "
            f"{row['session_excluded_fruit_balanced_mae']:.4f} d"
        ),
        (
            "  Delta MAE = "
            f"{row['delta_mae_days']:.4f} d"
        ),
        (
            "  MAE ratio = "
            f"{row['mae_retention_ratio']:.4f}"
        ),
        (
            "  R2 standard = "
            f"{row['standard_r2']:.4f}"
        ),
        (
            "  R2 session-excluded = "
            f"{row['session_excluded_r2']:.4f}"
        ),
        "",
    ])

review_lines.extend([
    "Interpretation rules:",
    "- Modest degradation under session exclusion supports robustness, not causality.",
    "- Severe degradation indicates dependence on same-day/session information and/or inability to interpolate/extrapolate.",
    "- Strong global-mean performance indicates an important amplitude component, but that component may still be physiological.",
    "- Day 0 and day 8 session-excluded results are extrapolation and must be interpreted separately.",
    "- Do not proceed to CNN before reviewing this package.",
])

REVIEW_PATH = (
    RESULTS /
    "ROBUSTNESS_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
